# tiny-yolov4-dronevehicle on dronevehicle — training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/dronevehicle/tiny-yolov4-dronevehicle_train.ipynb)

Role: training; tier **N** (4000 iterations by default, several hours on CPU); final evaluation on `SUBSET` images. Tasks: [T24.6](../../docs/tasks/M24-yolov4-tiny.md);
notebooks: [M14](../../docs/tasks/M14-notebooks.md).

Prerequisites:
- data: `data/dronevehicle/test/labels` — tools/get_datasets.sh dronevehicle (registration: source printed)
- weights: `weights/yolov4-tiny.weights` — tools/get_weights.sh

Each step prints the `python tools/…` command it runs: a result can be reproduced
from the command line. Outputs in `build/notebooks/dronevehicle/tiny-yolov4-dronevehicle/`, never in `results/` (an mAP on `SUBSET`
images is not publishable, M12 rules). Notebook generated by
`python -m tools.notebooks` (`make notebooks`): edit the template
`tools/notebooks/gabarits.py`, not this file.

## Parameters

In [ ]:
NET         = 'build/m11/cfg/tiny-yolov4-dronevehicle.cfg'  # name in CFG_FILES or cfg (relative to the root)
DATASET     = 'dronevehicle'  # key of DATASETS
SPLIT       = None  # None: evaluation split of the dataset
WEIGHTS     = 'build/notebooks/dronevehicle/tiny-yolov4-dronevehicle/final.weights'  # weights produced, evaluated at the end
DEVICE      = 'gpu'  # cpu | gpu (training only; the rest on CPU)
SUBSET      = 50  # images evaluated; 0: full split (tier N)
SIZE        = None  # input WxH (e.g. '640x192'); None: the one of the cfg
RESIZE      = 'stretch'  # stretch (M12 reference) | letterbox
METRIC      = None  # None: the one of the dataset (coco for coco and flir)
DATA_ROOT   = None  # dataset root; None: data/<dataset> (Colab: Drive folder)
BASE        = 'tiny-yolov4-coco'  # base of make_cfg.py
CHANNELS    = None  # channels of the cfg (1: thermal)
ANCHORS_K   = 6  # anchors of the cfg (6: YOLOv4-tiny)
INIT        = 'weights/yolov4-tiny.weights'  # coco | he | path of a .weights
INIT_NET    = 'tiny-yolov4-coco'  # network of the INIT weights if it differs from NET
ITERS       = 4000  # iterations (tier N above 600)
BATCH       = 16
LR          = 0.001
BURN_IN     = 500
MULTISCALE  = True  # 320-608 every 10 batches
SAVE_EVERY  = None  # None: default of train.py (200)
WORKERS     = 4
QAT         = ''  # wXaY scheme (e.g. 'w4a4'): QAT (T9.3, T12.5)
QAT_STEPS   = None  # steps.json of quant_lowbit.py (PTQ)
ADMM        = None  # JSON weight plan ({}: mixed6 everywhere, T9.2, T12.6)
ADMM_RHO    = 0.001
ADMM_GROWTH = 1.3
ADMM_EVERY  = 100
DRIVE_DIR   = '/content/drive/MyDrive/EmbeddedCV'  # Colab: archives data/<dataset>.tar, outputs and resume in runs/; None: no Drive
JOBS        = None  # evaluation processes; None: one per CPU core, capped by the memory (Colab TPU runtime: dozens)
SHOW        = 6  # images shown
OUT         = 'build/notebooks/dronevehicle/tiny-yolov4-dronevehicle'  # notebook outputs
REPO_URL    = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab: cloned repository
REV         = 'main'  # Colab: revision

## Environment

Local: nothing is installed or downloaded, the cell checks the prerequisites and
stops with the command to run. Colab: clone of the repository, installation, CuPy if
`DEVICE = "gpu"`, weights, then data by `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<dataset>.tar` or direct download; ExDark and FLIR: archive only,
made on the PC by `tools/get_datasets.sh pack <dataset>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone of an earlier session: reset to REV (notebook and code of the same commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules imported before the update
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # CPU runtime: no driver, CuPy would say "cudaErrorInsufficientDriver".
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("Colab runtime without a GPU: create a new Colab server "
                               "of GPU type (T4, L4 or A100), or DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # driver version
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("repository root not found: open the notebook from notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, INIT if str(INIT).endswith('.weights') else 'weights/yolov3-tiny.weights' if INIT == 'coco' else (), None, DATA_ROOT, COLAB, hint='',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Preparation (datasets other than VOC)

k-means anchors on the training split (`tools/kmeans_anchors.py`, table in `OUT`),
then a cfg with N classes (`tools/make_cfg.py`), as `tools/m11.sh <dataset>-prep`.
Skipped if the cfg already exists.

In [ ]:
import csv
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import estimate
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.models.tiny_yolo import load_cfg

if not str(NET).endswith(".cfg"):
    print(f"{NET}: cfg of the repository, no preparation")
elif Path(NET).exists():
    print(f"{NET} exists: preparation skipped")
else:
    anchors = Path(OUT) / f"anchors_{DATASET}{'_' + str(SIZE) if SIZE else ''}.md"
    C.run(C.cmd_anchors(DATASET, SIZE, anchors, DATA_ROOT))
    C.run(C.cmd_make_cfg(BASE, DATASET, C.anchors_k(anchors, ANCHORS_K), NET, SIZE,
                         CHANNELS))

## Preview of the targets

A few augmented images (`yolo.data.loader.VOCDataset`, hence `yolo.data.augment`)
with their boxes: a check of the annotations before hours of computation.

In [ ]:
cfg = load_cfg(NET)
size = input_size(cfg, parse_size(SIZE) if SIZE else None)
channels = cfg["input"][0]
names = datasets.DATASETS[DATASET].classes
samples = datasets.load(DATASET, DATA_ROOT, role="train")
print(f"{len(samples)} training images, input {size_label(size)}, {channels} channel(s)")
from yolo.data.loader import VOCDataset

preview = VOCDataset(samples[:SHOW], train=True, channels=channels)
for k in range(len(preview)):
    chw, boxes, labels = preview.load(k, size, np.random.default_rng(k))
    hwc = (np.clip(chw.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)
    img = Image.fromarray(hwc[..., 0] if channels == 1 else hwc)
    display(draw(img, boxes, np.ones(len(labels)), labels, names))

## Fine-tuning

`tools/train.py` in a subprocess whose output is relayed line by line; an
interruption of the kernel leaves the last `checkpoint.npz` (every `SAVE_EVERY`), and
running the cell again resumes (`--resume`). On Colab with `DRIVE_DIR`, `OUT` is
copied to `<DRIVE_DIR>/runs/` every 10 min and at the end of the cell
(`colab.sync_outputs`), then restored at the start of a session
(`colab.restore_outputs`): a cut session resumes from the last copied checkpoint, on
the same backend (T12.11, T14.8).

In [ ]:
import threading

from tools.notebooks import colab, runs

RUN = Path(OUT)
SYNC = COLAB and bool(DRIVE_DIR)
if SYNC:  # resume after a session cut: outputs of the previous session
    env.mount_drive(DRIVE_DIR)
    colab.restore_outputs(OUT, DRIVE_DIR)
RUN.mkdir(parents=True, exist_ok=True)
if ADMM and not Path(ADMM).exists():
    Path(ADMM).parent.mkdir(parents=True, exist_ok=True)
    Path(ADMM).write_text("{}\n")  # mixed6 everywhere, as tools/m12.sh admm
seconds = estimate(ITERS, BATCH, DEVICE, RUN / "loss.csv")
print(f"{ITERS} iterations × {BATCH} images: "
      + (f"≈ {seconds / 3600:.1f} h on {DEVICE}" if seconds else
         f"duration not measured on {DEVICE} (T12.11-e)"))
resume = (RUN / "checkpoint.npz").exists()
runs.write_meta(RUN, batch=BATCH, subset=0, iters=ITERS, lr=LR, device=DEVICE,
                net=NET, dataset=DATASET, qat=QAT, admm=bool(ADMM), rev=TRACE["rev"])
stop = threading.Event()


def sync_loop(every=600):  # copy to Drive every 10 min, checkpoint included
    while not stop.wait(every):
        colab.sync_outputs(OUT, DRIVE_DIR)


if SYNC:
    threading.Thread(target=sync_loop, daemon=True).start()
try:
    C.run(C.cmd_train(NET, RUN, DATASET, INIT, INIT_NET, resume, QAT, QAT_STEPS, ADMM,
                      ADMM_RHO, ADMM_GROWTH, ADMM_EVERY, ITERS, BATCH, LR, BURN_IN,
                      MULTISCALE, None, 0, SAVE_EVERY, WORKERS, DEVICE, DATA_ROOT),
          log=RUN / "log.txt")
finally:
    stop.set()
    if SYNC:
        colab.sync_outputs(OUT, DRIVE_DIR)

## Curves

Loss and learning rate of `loss.csv` (automatic mode of M13); running the cell again
during a long training redraws up to the last point written.

In [ ]:
C.run(C.cmd_figures("train", RUN))
for png in sorted((RUN / "figures").glob("*.png")):
    display(Image.open(png))

## Evaluation

`final.weights` by `tools/eval_voc.py` (`infer` template, T14.3); with QAT or ADMM,
the low-precision model by `tools/quant_lowbit.py`, then `tools/eval_quant.py`.

In [ ]:
if QAT or ADMM:
    C.run(C.cmd_quant_lowbit(NET, RUN / "checkpoint.npz", RUN / "model", QAT, bool(ADMM)))
    result = RUN / "map.json"
    C.run(C.cmd_eval_quant(NET, result, "int", SUBSET, JOBS, DATASET,
                           model_dir=RUN / "model", metric=METRIC, split=SPLIT,
                           size=SIZE, data_root=DATA_ROOT))
    d = json.loads(result.read_text())
    print(f"integer mAP {100 * d['results']['int']['map']:.2f} ({d['images']} images)")
else:
    result = RUN / "map_float.md"
    result.unlink(missing_ok=True)
    C.run(C.cmd_eval_voc(NET, RUN / "final.weights", DATASET, RESIZE, SUBSET, METRIC,
                         SPLIT, SIZE, DATA_ROOT, out=RUN / "dets", markdown=result,
                         jobs=JOBS))
    display(Markdown(result.read_text()))

## Backend speed (T12.11-e)

Seconds per image of the last iterations (`loss.csv`); the largest batch that fits
in GPU memory is found by running again with a growing `BATCH`. A number that comes
from weights trained on GPU names the backend.

In [ ]:
rows = list(csv.DictReader((RUN / "loss.csv").open())) if (RUN / "loss.csv").exists() else []
if rows:
    last = rows[-50:]
    s = sum(float(r["seconds"]) for r in last) / len(last) / BATCH
    print(f"{s:.3f} s/image on {DEVICE}, batch {BATCH}, {len(rows)} iterations logged")

## Summary

In [ ]:
print(f"outputs: {RUN}/ (checkpoint.npz, loss.csv, final.weights, figures/); "
      f"backend {DEVICE}; revision {TRACE['rev']}")
print(f"evaluation: {result}")
print("equivalent commands:")
for c in C.HISTORY:
    print("  " + c)